In [1]:
from pyspark.sql.functions import avg, count, round

In [ ]:
catalog = dbutils.widgets.get("catalog")  # Dynamically resolves to citibike_dev/test/prod, depending on which target this job was deployed to

In [ ]:
# Read from the Silver table using the dynamic catalog variable,
# so this notebook works correctly regardless of which environment (dev/test/prod) it runs in
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [3]:
# Gold aggregation — daily grouping, but broken down per station
# Only avg duration and total trips are kept here
df = df.groupBy("trip_start_date", "start_station_name").\
    agg(
        round(avg("trip_duration_mins"), 2).alias("avg_trip_duration_mins"),
        count("ride_id").alias("total_trips")
    )

In [ ]:
# Write the station-level daily aggregation as a managed Delta table in Gold,
# using the dynamic "catalog" variable
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.03_gold.daily_station_performance")